#Paper utilizado:

[1] M. Bahaghighat, M. Ghasemi y F. Ozen,
“A high-accuracy phishing website detection method based on machine learning,”
Journal of Information Security and Applications,
vol. 77,
art. 103553,
2023.
doi: 10.1016/j.jisa.2023.103553.
Available: https://www.sciencedirect.com/science/article/pii/S2214212623001370

#Limpieza de datos

En este colab se realizará la limpieza del set de datos para el entrenamiento de un modelo de inteligencia artificial. El dataset que se está usando es el de Abdelhamid, N. (2014). Website Phishing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5B301.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Import, según la documentación de UC Irvine

In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
website_phishing = fetch_ucirepo(id=379)

# data (as pandas dataframes)
X = website_phishing.data.features
y = website_phishing.data.targets

# metadata
print(website_phishing.metadata)

# variable information
print(website_phishing.variables)


{'uci_id': 379, 'name': 'Website Phishing', 'repository_url': 'https://archive.ics.uci.edu/dataset/379/website+phishing', 'data_url': 'https://archive.ics.uci.edu/static/public/379/data.csv', 'abstract': '\n\n', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1353, 'num_features': 9, 'feature_types': ['Integer'], 'demographics': [], 'target_col': ['Result'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2014, 'last_updated': 'Thu Mar 21 2024', 'dataset_doi': '10.24432/C5B301', 'creators': ['Neda Abdelhamid'], 'intro_paper': {'ID': 440, 'type': 'NATIVE', 'title': 'Phishing detection based Associative Classification data mining', 'authors': 'Neda Abdelhamid, A. Ayesh, F. Thabtah', 'venue': 'Expert systems with applications', 'year': 2014, 'journal': None, 'DOI': '10.1016/j.eswa.2014.03.019', 'URL': 'https://www.semanticscholar.org/paper/867e2293e9780b729705b4ba48d6b1

El dataset ya menciona que no hay valores faltantes, sin embargo aún así haré el análisis de valores faltantes junto con otros análisis para verificar la usabilidad de los datos más adelante.

Modificar la importación para manejarlo como pandas: ¿por qué? porque estoy más familiarizado y me gusta para analizar los datos.

In [ ]:
#pasar a pandas
X = pd.DataFrame(website_phishing.data.features)
y = pd.DataFrame(website_phishing.data.targets)

# estoy juntando el target con el resto de datos en una tabla (más adelante se vuelven a separar)
df = pd.concat([X, y], axis=1)

print(df.head())

   SFH  popUpWindow  SSLfinal_State  Request_URL  URL_of_Anchor  web_traffic  \
0    1           -1               1           -1             -1            1   
1   -1           -1              -1           -1             -1            0   
2    1           -1               0            0             -1            0   
3    1            0               1           -1             -1            0   
4   -1           -1               1           -1              0            0   

   URL_Length  age_of_domain  having_IP_Address  Result  
0           1              1                  0       0  
1           1              1                  1       1  
2          -1              1                  0       1  
3           1              1                  0       0  
4          -1              1                  0       1  


#Aqui podemos ver que tenemos 9 columnas de features y una de target que es la columna "Result"
También podemos observar que los valores son enteros que van de 0 a 1 y de -1 a 1.

Por ello asumo que los valores de 0 a 1 son booleanos. Y que las otras columnas siguen el mismo formato que Result donde los valores de -1 son categorizados como algo negativo, el 1 como algo positivo y el 0 como algo sospechoso.




In [ ]:
df.shape

(1353, 10)

Haré un .info() ya que este nos dice el tipo de dato de cada una de nuestras features.

Dos columnas distintas pueden tener tipos de datos distintos, sin embargo no pueden haber dos tipos de datos en una misma columna.

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1353 entries, 0 to 1352
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   SFH                1353 non-null   int64
 1   popUpWindow        1353 non-null   int64
 2   SSLfinal_State     1353 non-null   int64
 3   Request_URL        1353 non-null   int64
 4   URL_of_Anchor      1353 non-null   int64
 5   web_traffic        1353 non-null   int64
 6   URL_Length         1353 non-null   int64
 7   age_of_domain      1353 non-null   int64
 8   having_IP_Address  1353 non-null   int64
 9   Result             1353 non-null   int64
dtypes: int64(10)
memory usage: 105.8 KB
None


Todas las columnas tienen un solo tipo de dato (int64). Por ello podemos asumir que no hay anomalías en nuestro set de datos en el tipo de dato registrado. De lo contrario, se debería de cambiar el tipo de dato de las filas mal registradas en caso de ser posible o eliminar las filas (En este problema no veo muy recomendable rellenar espacios faltantes con valores aproximados o promedio ya que son clasificaciones todas las columnas).

#El siguiente describe() nos da estadísticas generales del set de datos. Esto nos ayuda a detectar otras anomalías en los datos: Por ejemplo cuando hay un valor muy grande que pueda afectar los cálculos.

Como mencioné, parece que en este dataset todos los datos son binarios o son -1, 0 y 1. Por ello los mínimos y máximos no deberían de exceder estos valores.

En realidad esperamos que todos los valores se encuentren en estos rangos excepto por la cuenta en lo que esperamos que todos los valores tengan la misma cuenta que el df.shape. Si es menor significa que tenemos valores faltantes y debemos de decidir cómo tratarlos.

Me parece curioso que en age of domain no tengamos valores enteros sino ya la interpretación de si el sitio es sospechosamente reciente.

De igual forma me parece que el único booleano del set de datos es el de si tiene una dirección IP donde asumo que el 1 representa algo malo pues no hay un dominio.

In [ ]:
df.describe()

,SFH,popUpWindow,SSLfinal_State,Request_URL,URL_of_Anchor,web_traffic,URL_Length,age_of_domain,having_IP_Address,Result
count,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000,1353.000000
mean,0.237990,-0.258684,0.327421,-0.223208,-0.025129,0.000000,-0.053215,0.219512,0.114560,-0.113821
std,0.916389,0.679072,0.822193,0.799682,0.936262,0.806776,0.762552,0.975970,0.318608,0.954773
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,-1.000000
25%,-1.000000,-1.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,-1.000000
50%,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,-1.000000
75%,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


A partir de la tebla previa podemos asumir que todos los valores se encuentran bien registrados y que no contamos con valores faltantes.

Aún así, realizaré un script de limpieza simulando que sí hay valores faltantes en las columnas. Me parece valioso tener el código que sirve para revisar y corregir si hay valores faltantes

#Limpieza de datos



In [ ]:
conteo_no_nulos = df.count().sort_values(ascending=False)
total_filas = len(df)

resumen_completitud = pd.DataFrame({
    "No_nulos": conteo_no_nulos,
    "Total_filas": total_filas,
    "Porcentaje_completitud (%)": round(conteo_no_nulos / total_filas * 100, 2)
})

resumen_completitud = resumen_completitud.sort_values(by="Porcentaje_completitud (%)", ascending=False)

print("Resumen de completitud de columnas:")
display(resumen_completitud)

umbral = 70
columnas_bajas = resumen_completitud[resumen_completitud["Porcentaje_completitud (%)"] < umbral]
if not columnas_bajas.empty:
    print(f"\n Columnas con menos del {umbral}% de completitud:")
    display(columnas_bajas)
else:
    print(f"\nTodas las columnas tienen al menos {umbral}% de completitud.")


Resumen de completitud de columnas:


,No_nulos,Total_filas,Porcentaje_completitud (%)
SFH,1353,1353,100.0
popUpWindow,1353,1353,100.0
SSLfinal_State,1353,1353,100.0
Request_URL,1353,1353,100.0
URL_of_Anchor,1353,1353,100.0
web_traffic,1353,1353,100.0
URL_Length,1353,1353,100.0
age_of_domain,1353,1353,100.0
having_IP_Address,1353,1353,100.0
Result,1353,1353,100.0



Todas las columnas tienen al menos 70% de completitud.


En el código previo se marcan las columnas que tengan menos del 70% de sus filas llenas ya que pueden llegar a ser problemáticas para determinar si vale la pena eliminar las filas o directamente la columna.


##Identificar valores únicos

El siguiente código puede ser medio redundante porque describe() ya me dice los valores únicos de las columnas de tipo object.

Sin embargo el código me permite hacer lo mismo incluso cuando no son objetos.

Con esto puedo identificar si alguna columna no me es útil (cuando todas o casi todas las instancias tienen el mismo valor)

In [ ]:
dfejemplo = pd.DataFrame({
    "color": ["rojo", "azul", "rojo", "verde"]
})

print(dfejemplo.describe(include='object'))

       color
count      4
unique     3
top     rojo
freq       2


In [ ]:
valores_unicos = df.nunique().sort_values()

resumen_unicos = pd.DataFrame({
    "Valores_únicos": valores_unicos,
    "Total_filas": len(df),
    "Porcentaje_únicos (%)": round(valores_unicos / len(df) * 100, 3)
})

# Mostrar de menor a mayor (columnas menos variables primero)
print("Cantidad de valores únicos por columna:")
display(resumen_unicos)

# === (Opcional) Mostrar solo columnas con muy poca variabilidad ===
umbral_unicos = 2  # por ejemplo, columnas con solo 1 o 2 valores únicos
columnas_poco_utiles = resumen_unicos[resumen_unicos["Valores_únicos"] <= umbral_unicos]

if not columnas_poco_utiles.empty:
    print(f"\n !!!!!!!!!!!!! Columnas con {umbral_unicos} o menos valores únicos:")
    display(columnas_poco_utiles)
else:
    print(f"\n Todas las columnas tienen más de {umbral_unicos} valores únicos.")


Cantidad de valores únicos por columna:


,Valores_únicos,Total_filas,Porcentaje_únicos (%)
age_of_domain,2,1353,0.148
having_IP_Address,2,1353,0.148
SSLfinal_State,3,1353,0.222
SFH,3,1353,0.222
Request_URL,3,1353,0.222
URL_of_Anchor,3,1353,0.222
web_traffic,3,1353,0.222
popUpWindow,3,1353,0.222
URL_Length,3,1353,0.222
Result,3,1353,0.222



 !!!!!!!!!!!!! Columnas con 2 o menos valores únicos:


,Valores_únicos,Total_filas,Porcentaje_únicos (%)
age_of_domain,2,1353,0.148
having_IP_Address,2,1353,0.148


Algo interesante que sale de aquí es que age of domain solo tiene dos valores a pesar de no ser un booleano.

##limpieza de columnas

El siguiente código sirve para eliminar columnas que sean muy problemáticas o que no aporten valor

In [ ]:
df.shape

(1353, 10)

In [ ]:
# Lista de columnas a eliminar, No hay que eliminar ninguna
columnas_a_eliminar = [
    "Nombre de columna a eliminar"
]

# Eliminar las columnas (si existen)
df_limpio = df.drop(columns=[col for col in columnas_a_eliminar if col in df.columns])

print("Columnas eliminadas correctamente.")

df_limpio.shape


Columnas eliminadas correctamente.


(1353, 10)

##Preparación para xgboost

Como es un problema de clasificación, estoy pensando en la posibilidad de utilizar xgboost. Sin embargo, a xgboost no le gusta que sea -1, 0, 1. Por lo que se cambia a 0, 1, 2. Esto no afecta el resultado de ninguno de los modelos ya que solo estamos dando valores distintos a una clasificación.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df_limpio["Result"] = label_encoder.fit_transform(df_limpio["Result"])

# En el siguiente código con dropna, se eliminan las filas con valores vacios. En este caso no aplica ya que no hay valores vacios

In [ ]:
filas_antes = len(df_limpio)

df_limpio = df_limpio.dropna()

filas_despues = len(df_limpio)
filas_eliminadas = filas_antes - filas_despues

print(f" Se eliminaron {filas_eliminadas} filas con datos vacíos.")
print(f" Total antes: {filas_antes} → Total después: {filas_despues}")

df_limpio.to_excel("/content/WebPhish.xlsx", index=False)
print("Archivo final guardado como 'WebPhish.xlsx'")

 Se eliminaron 0 filas con datos vacíos.
 Total antes: 1353 → Total después: 1353
Archivo final guardado como 'WebPhish.xlsx'


In [ ]:
df_limpio.shape

(1353, 10)

In [ ]:
import joblib

joblib.dump(label_encoder, "label_encoder.pkl")

['label_encoder.pkl']